<a href="https://colab.research.google.com/github/climatom/BioMetConference/blob/main/notebooks/01_HumidHeatMetrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Humid Heat Metrics

## Introduction

In this workbook we will familiarise ourselves with some key variables that can be used to characterise, and understand, extreme heat events. Our focus here will be two very useful measures:

* (1) Wet-bulb temperature ($T_w$)
* (2) Equivalent temperature ($T_e$), including its 'latent' component

By the end of this notebook you should be able to:

- explain the difference between air temperature and wet-bulb temperature;
- interpret maps of humid heat;
- distinguish between thermal and moisture contributions to humid heat;
- explain why equivalent temperature is useful for understanding humid-heat events.

We will learn about these by working through a case study of the June 2026 heatwave in the UK. We begin by characterising the heatwave in terms of $T_w$. We then consider how $T_e$ can help us identify the drivers of the event. Finally, we will reflect on the importance of using measures of humid heat in the context of understanding extreme events.

## Wet-bulb temperature

The *thermodynamic* (or 'isobaric') wet-bulb temperature represents "the temperature an air parcel would have if cooled adiabatically to saturation at constant pressure by evaporation of water into it, all latent heat being supplied by the parcel" [(AMS, 2026)](https://glossary.ametsoc.org/wiki/wet-bulb-temperature/). It can be expressed by the implicit equation:


$$c_{pm} T_a + L_v q = c_{pm} T_w + L_v q_s(T_w,p)\tag{1}$$



where $c_{pm}$ is the specific heat capacity of moist air, $L_v$ is the latent heat of vaporisation, $q$ is the specific humidity, and the subscript $s$ denotes the saturation specific humidity evaluated at the wet-bulb temperature and barometric pressure ($T_w$, $p$).

The intuition here is that the moist enthalpy of the unsaturated air parcel (left-hand side) equals the moist enthalpy of a saturated air parcel at the wet-bulb temperature (right-hand side):


$$c_{pm} (T_a-T_w) + L_v [q - q_s(T_w,p)] = 0\tag{2}$$


Since $q_s$ depends non-linearly on $T_w$ this equation cannot be solved analytically and must, instead, be solved through iteration. Many suitable approaches exist (e.g., [Romps (2026)](https://journals.ametsoc.org/view/journals/apme/65/2/JAMC-D-25-0130.1.xml)).


## June 2026 UK Heatwave

### Data

Here, we use [ERA5 Land](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land?tab=overview) to assess this recent event. We select this dataset because of its low latency, high resolution, and complete spatial coverage. A major limitation is that whilst ERA5 Land is observationally-constrained, it is not purely an observational data product.

Specifically, hourly $T_a$, $p$ and dewpoint temperature ($T_d$) (1950-2026) were downloaded from the Copernicus Climate Data Store using the Python API. A copy of this code is available [here](https://drive.google.com/file/d/1FuRmXcbqJj1E0VCdu5oWlQGsZ-ymLw01/view?usp=sharing). Note that, to run it (after editing), [you would need to setup the Climate Data Store API
](https://cds.climate.copernicus.eu/how-to-api). **You do not need to do this for the workshop**.

### Data pre-processing

ERA5 products (including Land) do not distribute near-surface $q$. Instead, it must be computed from dewpoint temperature ($T_d$) and $p$:

$$q =\frac{0.622\,e_s(T_d)}{p-0.378\,e_s(T_d)}$$

in which $e_s$ is the saturation vapour pressure computed after [Huang (2018)](https://journals.ametsoc.org/view/journals/apme/57/6/jamc-d-17-0334.1.xml).

Subsequently, a variety of moist heat metrics have been computed for the workshop (including $T_w$). Useful libraries for computing these yourselves include [MetPy](https://unidata.github.io/MetPy/latest/index.html) and [Thermofeel](https://unidata.github.io/MetPy/latest/index.html).

### Our assessment

We are now ready to start exploring the event -- including running the code! The mechanics of this part are *very* straightforward. Code can be run by either hovering over the top left of a code cell (like the one below), and pressing the 'play' button; or you can use the keyboard shortcut (ctrl + enter)after having clicked in the cell; or you can click in the cell and then choose 'Run selection' under the Runtime menu at the top of your browser window.

**Run the code cell below** to set this notebook up for our analysis. This first cell installs the relevant python libraries that we will use, and it also configures the notebook to read in data correctly.

*Note that code is not displayed in the default view of this notebook (to help improve readability). You can view the code by clicking on the 'show code' button (otherwise just click play to run).*

In [ ]:
# @title
#### Configuration

# Installations
!pip -q install \
    cartopy \
    xarray \
    netCDF4 \
    gdown


# Imports
import gdown
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path


# Configure access to data directory
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(exist_ok=True)
## Data dir url
data_url="https://drive.google.com/drive/folders/1DiiIUnestnkalfct9p58lSn_HsRc16GX?usp=sharing"
## Download data (< 1GB) into user's drive folder
gdown.download_folder(
    data_url,
    output=str(DATA_DIR),
    quiet=False,
    use_cookies=False,
)




The code below reads in June 2026 daily maxima for $T_a$ and $T_w$ 2026, computes the time maxima, and plots this in two panels. Run it, and please see the comments in the code (anything behind '#') for further information about what each line is doing.

In [ ]:
# @title
# First, we use xarray to open the files

# June daily maxima
jun26 = xr.open_dataset("/content/data/era5land_uk_dailymax_Jun2026_thermo.nc")
# Annual maxima
annmax = xr.open_dataset("/content/data/era5land_uk_annmax_Jun-Aug_thermo.nc")
# Climatology (mean of June-August daily maxima)
clim = xr.open_dataset("/content/data/era5land_uk_1981-2010_clim_thermo.nc")
# Clip the data to minimise the influence of NW France
LAT_MIN, LAT_MAX = 50.0, 61.0
LON_MIN, LON_MAX = -11.0, 2.0
def clip(ds):
    return ds.sel(
        latitude=slice(LAT_MAX, LAT_MIN),   # ERA5 latitude is descending
        longitude=slice(LON_MIN, LON_MAX)
    )
jun26 = clip(jun26)
annmax = clip(annmax)
clim = clip(clim)


# We can print the files to ascertain their structure--illustrated below just
# for the jun26 dataset. Toggle this off (by commenting) if you want to suppress
# the output
print(jun26)

# Some of the data variables will be mysterious (for now); but you should
# recognise ta and tw. Note, too, that (reasssuringly) the time coordinate is
# 30 days in length!

# Plotting
# ============================================================
# June 2026 daily maximum air and wet-bulb temperatures
# ============================================================

# Take the time max
zplot = jun26.max("valid_time")

# Set up the plot
fig, axes = plt.subplots(
    1, 2,
    figsize=(10, 5),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True,
)

for ax in axes:
    ax.coastlines(resolution="10m", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Air temperature
pcm = axes[0].pcolormesh(
    jun26.longitude,
    jun26.latitude,
    zplot.ta-273.15,
    transform=ccrs.PlateCarree(),
    cmap="hot_r",
)
axes[0].set_title("Air temperature")

cb = fig.colorbar(
    pcm,
    ax=axes[0],
    orientation="horizontal",
    pad=0.05,
)

cb.set_label("Temperature [°C]")

# Wet-bulb temperature
pcm = axes[1].pcolormesh(
    jun26.longitude,
    jun26.latitude,
    zplot.tw-273.15,
    transform=ccrs.PlateCarree(),
    cmap="hot_r",
)

axes[1].set_title("Wet-bulb temperature")

cb = fig.colorbar(
    pcm,
    ax=axes[1],
    orientation="horizontal",
    pad=0.05,
)

cb.set_label("Temperature [°C]")

plt.show()

Noting that the colour ramps are *not* the same, what do you notice about the relative magnitudes of $T_a$ and $T_w$?

[**Going further**... If you have experience with Python/xarray, try computing the mean wet-bulb depression ($T_a - T_w$). You can do this within the notebook by adding a new code cell (**+ Code**). Depending on your settings, Google's Gemini AI assistant might also prove useful.]


## Was the June 2026 heat event more extreme for $T_a$ or $T_w$?

Unless the air is completely saturated, $T_w$ will always be appreciably lower than $T_a$. Hence, the public may struggle to relate forecast or reported $T_w$ to expected impacts. For example, during the 2003 heatwave---the deadliest on record---$T_w$ remained well below $30^\circ$C  ([Raymond et al., 2020](https://www.science.org/doi/10.1126/sciadv.aaw1838)).

We will now assess how unprecedented the peak 2026 heatwave was for $T_a$ and $T_w$ using their ranks relative to the annual maxima. Specifically, the code below computes ranks (1 = highest since 1950). It then plots these ranks and annotates them with the area-weighted mean fraction of record-breaking heat.
Cells experiencing their record in 2026 are also stippled.

In [ ]:
# @title
target = jun26[["ta", "tw"]].max("valid_time")

valid_mask = (
    np.isfinite(target["tw"]) &
    np.isfinite(annmax["tw"]).any("year")
)

rank = xr.Dataset()

for var in ["ta", "tw"]:
    rank[var] = (
        1 + (annmax[var] > target[var]).sum("year")
    ).where(valid_mask)

# 2D latitude weights
weights2d = xr.ones_like(rank.ta) * np.cos(np.deg2rad(rank.latitude))

denom = weights2d.where(valid_mask).sum(("latitude", "longitude"))

frac_record = xr.Dataset()

for var in rank.data_vars:
    frac_record[var] = (
        weights2d.where((rank[var] == 1) & valid_mask)
        .sum(("latitude", "longitude"))
        / denom
    )



In [ ]:
# @title
# Plotting

# Set low upper limit to help illuminate detail
rank_plot = rank.where(rank <= 10)

fig, axes = plt.subplots(
    1, 2,
    figsize=(10, 5),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True,
)

for ax in axes:
    ax.coastlines(resolution="10m", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Air temperature rank
pcm = axes[0].pcolormesh(
    rank.longitude,
    rank.latitude,
    rank_plot.ta,
    transform=ccrs.PlateCarree(),
    cmap="magma_r",
    vmin=1,
    vmax=10,
)

# Stipple/scatter rank == 1 cells
ta_record = rank.ta == 1
lon2d, lat2d = np.meshgrid(rank.longitude, rank.latitude)

axes[0].scatter(
    lon2d[ta_record.values],
    lat2d[ta_record.values],
    s=0.5,
    c="k",
    marker=".",
    alpha=0.3,
    transform=ccrs.PlateCarree(),
)

axes[0].set_title(
    f"$T_a$ record fraction: {100*frac_record.ta.item():.1f}%"
)

# Wet-bulb temperature rank
pcm = axes[1].pcolormesh(
    rank.longitude,
    rank.latitude,
    rank_plot.tw,
    transform=ccrs.PlateCarree(),
    cmap="magma_r",
    vmin=1,
    vmax=10,
)

# Stipple/scatter rank == 1 cells
tw_record = rank.tw == 1

axes[1].scatter(
    lon2d[tw_record.values],
    lat2d[tw_record.values],
    s=0.5,
    c="k",
    marker=".",
    alpha=0.3,
    transform=ccrs.PlateCarree(),
)

axes[1].set_title(
    f"$T_w$ record fraction: {100*frac_record.tw.item():.1f}%"
)

# Shared colourbar
cb = fig.colorbar(
    pcm,
    ax=axes,
    orientation="horizontal",
    pad=0.05,
    shrink=0.6,
)

cb.set_label("Rank [1 = hottest on record; only top 10 shown]")

plt.show()

# Understanding extreme $T_w$ drivers #

This was clearly an exceptional $T_w$ event for the UK. June is not, of course, typically the hottest month, so the record-breaking observed above is even more unusual. But what caused it? To probe this, we will introduce another key metric -- the equivalent temperature.

## Equivalent temperature

$T_w$ is a useful measure for quantifying humid heat because it combines the effects of air temperature and atmospheric moisture into a single variable. However, this also makes it difficult to determine *why* a particular event was so extreme. Was it unusually hot? Exceptionally humid? Or both?

To answer these questions, the **equivalent temperature** ([Matthews et al. 2022](https://wires.onlinelibrary.wiley.com/doi/10.1002/wcc.779)) ($T_e$) can be very helpful:

$$ T_e = T_a + T_q\tag{3}$$

where

$$T_q = \frac{L_v q}{c_{pm}}\tag{4}$$

is the **latent temperature**: the temperature increase that would result if all of the energy stored as water vapour were converted into sensible heat.

$T_e$ is therefore a measure of the **total moist enthalpy** of an air parcel, expressed in units of temperature. Unlike $T_w$, it separates naturally into two physically meaningful components:

- $T_a$: the sensible (dry) heat contribution;
- $T_q$: the latent (moisture) contribution.

This decomposition allows us to answer a simple but important question:

> Was the June 2026 humid heat event primarily driven by unusually high air temperatures, unusually high atmospheric moisture, or a combination of the two? The $T_e$ framework is particularly useful to answer this question because thermodynamic wet-bulb temperature is *equally sensitive* to changes in $T_a$ and $T_q$. Comparing anomalies in these two quantities therefore provides a physically meaningful decomposition of the drivers of extreme humid heat.

In the following, we will compare anomalies in $T_a$ and $T_q$ to understand the relative importance of these two ingredients.




In [ ]:
# @title
# Compute Ta and Tq anomalies from the LTM
## Import a new file here. This has the max ta and tq (and te) on the day of
## the maximum tw in 2026.
thermo_comps=xr.open_dataset("/content/data/annual_max_thermo_at_max_tw.nc")
thermo_comps26=thermo_comps.sel(year=2026)
thermo_comps=clip(thermo_comps)
thermo_comps26=clip(thermo_comps26)

# Subtract these peak ta and tq on the hottest tw day from their long-term maxima
ta_anom=thermo_comps26.ta_at_tw_max-clim.ta
tq_anom=thermo_comps26.tq_at_tw_max-clim.tq


In [ ]:
# @title
# Plotting
vmi=-20
vma=20

fig, axes = plt.subplots(
    1, 2,
    figsize=(10, 5),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True,
)

for ax in axes:
    ax.coastlines(resolution="10m", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Air temperature anomaly
pcm = axes[0].pcolormesh(
    ta_anom.longitude,
    ta_anom.latitude,
    ta_anom.values,
    transform=ccrs.PlateCarree(),
    cmap="RdYlBu_r",
    vmin=vmi,
    vmax=vma,
)

axes[0].set_title(
    "$T_a$ anomaly"
)

# Latent temperature anomaly
pcm = axes[1].pcolormesh(
    tq_anom.longitude,
    tq_anom.latitude,
    tq_anom.values,
    transform=ccrs.PlateCarree(),
    cmap="RdYlBu_r",
    vmin=vmi,
    vmax=vma,
)

axes[1].set_title(
    f"$T_q$ anomlay"
)

# Shared colourbar
cb = fig.colorbar(
    pcm,
    ax=axes,
    orientation="horizontal",
    pad=0.05,
    shrink=0.6,
)

cb.set_label("Temperature anomlay [$^{circ}$C]")

plt.show()

## Interpretation

As even a quick visual comparison of the plots shows, it was $T_q$ (i.e., the *humidity*) that was remarkable in driving up $T_w$!


Which techniques could we use to assess what caused the $T_q$ anomaly?

**Working in pairs, discuss at least two hypotheses that could explain the humidity anomaly. What observations, datasets, and methods would you need to distinguish between them?**.

A related question, that helps illustrate the utility of tracking $T_w$ (or $T_e$) (and might help prompt some answers to the above), is *why* did the southeast of England fall some way short of its $T_a$ record--much more so than surrounding areas (as seen above in the ranking plots)?

One possible explanation is the thunderstorms that dramatically [swept the capital on the night of the 22-23 June](https://www.bbc.co.uk/news/articles/ckg84g3zv7po). Below, we plot those locations that experienced more than 5 mm precipitation (according to the UK [Met Office NIMROD rainfall dataset](https://catalogue.ceda.ac.uk/uuid/82adec1f896af6169112d09cc1174499/). See how the strong suppression of $T_a$ coincides with the band of rainfall over London (shown with the cyan dot).

But, also note that  $T_w$ is *not* so similarly dampened.

## Some more questions for discussion:

*   Why would (overnight) precipitation suppress $T_a$, *especially in Southeast England* (where conditions are generally drier).
*   Why might we *expect* $T_w$ to be *less* affected?
* What might we have concluded about the June 2026 event if we had analysed only $T_a$? What additional insight do $T_w$ and $T_e$ provide?
* Are there any learnings here relevant for climate change adaptation?



In [ ]:

# @title
# Plotting
STIPPLE_SIZE=0.5
ALPHA=0.75

# Read in Nimrod rainfall data
pr=xr.open_dataset("/content/data/nimrod_22-26_June.nc")
pr=pr.sel(valid_time=slice("2026-06-22","2026-06-23"))
pr_tot = pr.nimrod_daily_rainfall.sum("valid_time", skipna=True)

# Plot
fig, axes = plt.subplots(
    1, 2,
    figsize=(10, 5),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True,
)

for ax in axes:
    ax.coastlines(resolution="10m", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

# Air temperature rank
pcm = axes[0].pcolormesh(
    rank.longitude,
    rank.latitude,
    rank_plot.ta,
    transform=ccrs.PlateCarree(),
    cmap="inferno_r",
    vmin=1,
    vmax=10,
)

axes[0].set_title(
    f"$T_a$")

# Wet-bulb temperature rank
pcm = axes[1].pcolormesh(
    rank.longitude,
    rank.latitude,
    rank_plot.tw,
    transform=ccrs.PlateCarree(),
    cmap="inferno_r",
    vmin=1,
    vmax=10,
)
axes[1].set_title(
    f"$T_w$")

# Shared colourbar
cb = fig.colorbar(
    pcm,
    ax=axes,
    orientation="horizontal",
    pad=0.05,
    shrink=0.6,
)

cb.set_label("Rank [1 = hottest on record; only top 10 shown]")


#pr_stipple = pr_tot_box.where(pr_tot_box >= pr_k)]
pr_stipple = pr_tot.where(pr_tot >=5)

# Convert mask to 1D lat/lon coordinates for scatter
stipple_df = (
    pr_stipple
    .rename("pr")
    .to_dataframe()
    .reset_index()
    .dropna(subset=["pr"])
)

# Plot the precipitation
for ax in axes:
  ax.scatter(
              stipple_df["lon"],
              stipple_df["lat"],
              s=STIPPLE_SIZE,
              marker=".",
              color="blue",
              alpha=ALPHA,
              transform=ccrs.PlateCarree(),
              label=f"Precip ≥ 10 mm)",
              zorder=5,
          )
  # Plot london
  ax.scatter(-0.1278,51.5074,
             transform=ccrs.PlateCarree(),
             s=22,
             color='cyan')
plt.show()